# Lab 5: Registro do Baseline — Naive RAG para Acórdãos do TCU

## Objetivos da Aula

Este laboratório estabelece o **baseline Naive RAG** sobre o corpus de **Acórdãos do TCU (2026)** que será usado em comparações ao longo de TODAS as aulas subsequentes (aulas 3-12). Sem um baseline reproduzível, não é possível medir progresso em sistemas RAG.

### Três objetivos principais:
1. **Executar 5 queries de teste** cobrindo diferentes tipos de necessidade informacional sobre jurisprudência de controle externo
2. **Documentar respostas e artefatos** (chunks recuperados, tokens, tempos, métricas de recuperação Hit@1 / Hit@K / MRR)
3. **Avaliar qualitativamente** nas 5 dimensões que importam em RAG jurídico aplicado ao TCU

---

## Domínio do Baseline

| Item | Valor |
|---|---|
| Corpus | Acórdãos do TCU — ano 2026 |
| Volume | 5.998 acórdãos (CSV `acordao-completo-2026.csv`) |
| Tipos de processo | TCE, REPR, MON, RA, APOS, PCIV, PMIL e outros |
| Colegiados | Plenário, 1ª Câmara, 2ª Câmara |
| Campo indexado | `ACORDAO` (texto completo com tags HTML removidas) |
| Identificador único | `KEY` (ex.: `ACORDAO-COMPLETO-2765836`) |

> **Observação didática:** este notebook inicia o baseline com um corpus **reduzido aos 5 documentos TOP-1 esperados** (cenário ideal). Depois de validar o pipeline, o aluno é convidado a indexar o CSV completo e observar a queda de Hit@1 — é exatamente essa queda que o curso vai atacar com reranking, query expansion, HyDE etc.

---

## Diagrama: O Baseline como Âncora

```
                         AULA 2: LAB 5
                  Baseline Naive RAG (TCU)
                       (v0.0 do sistema)
                            |
                 ___________+___________
                 |         |           |
            AULA 3      AULA 5      AULA 7
          Reranking   Compressão  Query Exp.
                 |         |           |
                 |    AULA 8: Self-RAG |
                 |         |           |
                 |    AULA 10: HyDE    |
                 |         |           |
                 +_________+___________+
                           |
                    Resultado final:
                  Melhoria documentada
                 (baseline → v1.0 final)
```

---

## Referências ABNT

- LEWIS, P. et al. **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks**. *NeurIPS*, 2020. Disponível em: https://arxiv.org/abs/2005.11401
- GAO, Y. et al. **Retrieval-Augmented Generation for Large Language Models: A Survey**. arXiv, 2023. Disponível em: https://arxiv.org/abs/2312.10997
- BRASIL. Tribunal de Contas da União. **Regimento Interno do TCU**. Brasília: TCU.
- BRASIL. **Lei nº 8.443, de 16 de julho de 1992** (Lei Orgânica do TCU).


In [ ]:
# Célula 1: Instalação de dependências e configuração dos Provedores (Groq como Opção 1, Ollama como Opção 2)

import subprocess, sys

packages = [
    'langchain>=0.3',
    'langchain-community>=0.3',
    'langchain-core>=0.3',
    'langchain-ollama>=0.2',     # cliente nativo do Ollama (LLM + Embeddings)
    'langchain-openai>=0.2',     # alternativa via endpoint OpenAI-compat do Ollama
    'sentence-transformers>=3.0',# fallback HuggingFace para BGE-M3
    'faiss-cpu>=1.8',
    'opensearch-py>=2.7',
    'pandas>=2.2',
    'openpyxl>=3.1',
    'matplotlib>=3.9',
    'seaborn>=0.13',
    'jinja2>=3.1',
    'numpy>=1.26',
    'scikit-learn>=1.5',
    'tqdm>=4.66',
    'python-dotenv>=1.0',
    'requests>=2.32',
    'groq>=0.5.0',               # 🟢 SDK oficial do Groq
    'langchain-groq>=0.2.0'      # 🟢 Integração do Groq com LangChain
]

print('\n' + '='*60)
print('INSTALAÇÃO DE DEPENDÊNCIAS — PIPELINE RAG (Groq + Ollama)')
print('='*60 + '\n')

for package in packages:
    print(f'Instalando {package}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('\n' + '='*60)
print('VERIFICAÇÃO DE VERSÕES')
print('='*60 + '\n')

import langchain
from langchain_community.vectorstores import FAISS
import pandas as pd, numpy as np, matplotlib, seaborn as sns

print(f'✓ langchain: {langchain.__version__}')
print(f'✓ pandas: {pd.__version__}')
print(f'✓ numpy: {np.__version__}')
print(f'✓ matplotlib: {matplotlib.__version__}')
print(f'✓ seaborn: {sns.__version__}')
print(f'\n✓ Python: {sys.version.split()[0]}')
print('\n' + '='*60)
print('INSTALAÇÃO CONCLUÍDA')
print('='*60)

# ─── Configuração e Verificação dos Provedores de LLM ───
import os, requests
from dotenv import load_dotenv
from pathlib import Path
load_dotenv()

# Carregar variáveis do .env
GROQ_API_KEY = os.getenv('GROQ_API_KEY', '')
GROQ_LLM_MODEL = os.getenv('GROQ_LLM_MODEL', 'llama-3.1-8b-instant')

OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
OLLAMA_LLM_MODEL = os.getenv('OLLAMA_LLM_MODEL', 'llama3.1:8b')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL', 'bge-m3')

# 👉 ALUNO: caminho do CSV de acórdãos TCU 2026
DATASET_DIR = Path("../datasets").resolve()
CSV_PATH = DATASET_DIR / "acordao-completo-2026.csv"

# Variáveis globais que definirão o comportamento das próximas células
PROVEDOR_ATIVO = None
MODELO_LLM_ATIVO = None

print('\n' + '='*60)
print('🔍 CHECAGEM DE PROVEDORES DE LLM')
print('='*60)

# 🚀 OPÇÃO 1: Tentar configurar o Groq na Nuvem
print('\n⚡ Testando Opção 1: Groq (Nuvem)...')
if GROQ_API_KEY:
    try:
        from groq import Groq
        client = Groq(api_key=GROQ_API_KEY)
        client.models.list()

        PROVEDOR_ATIVO = 'groq'
        MODELO_LLM_ATIVO = GROQ_LLM_MODEL
        print(f'   ✅ Groq autenticado com sucesso!')
        print(f'   🎯 Modelo selecionado: {MODELO_LLM_ATIVO}')
    except Exception as e:
        print(f'   ⚠️ Chave do Groq encontrada, mas a conexão falhou: {e}')
else:
    print('   ℹ️ GROQ_API_KEY não encontrada no arquivo .env.')

# 🐳 OPÇÃO 2: Fallback / Teste do Ollama Local
print('\n🐳 Testando Opção 2: Ollama (Local)...')
ollama_disponivel = False
try:
    r = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=3)
    modelos_ollama = [m['name'] for m in r.json().get('models', [])]
    ollama_disponivel = True
    print(f'   ✅ Ollama respondendo em {OLLAMA_BASE_URL}')
    print(f'   📦 Modelos locais instalados: {modelos_ollama}')
except Exception as e:
    print(f'   ❌ Ollama indisponível em {OLLAMA_BASE_URL} ({e})')

# ⚖️ Decisão do Provedor Ativo do Pipeline
print('\n' + '-'*60)
if PROVEDOR_ATIVO == 'groq':
    print(f'🏆 PROVEDOR DEFINIDO: [ GROQ ] será usado como LLM principal.')
else:
    if ollama_disponivel:
        PROVEDOR_ATIVO = 'ollama'
        MODELO_LLM_ATIVO = OLLAMA_LLM_MODEL
        print(f'🏆 PROVEDOR DEFINIDO: [ OLLAMA ] será usado (Fallback acionado).')
    else:
        print(f'❌ ERRO CRÍTICO: Nenhum dos provedores (Groq ou Ollama) está pronto para uso!')
print('-'*60 + '\n')
print(f'📄 CSV de acórdãos configurado: {CSV_PATH}')


## Conceitos: O que é um Bom Baseline RAG?

Um baseline deve ter **4 propriedades críticas**:

### 1. **Reproduzível**
- Mesmos dados de entrada → mesmos resultados
- Não depende de API externa não-determinística
- Seeds fixadas para embeddings e geração (temperatura = 0)

### 2. **Documentado**
- Versão exata de cada componente (modelo, LLM, índice)
- Hiperparâmetros explícitos (k=5, chunk_size=512, etc)
- Todos os passos registrados

### 3. **Auditável**
- Cada decisão é rastreável até sua fonte
- Chunks recuperados visíveis e avaliáveis
- Scores de similaridade capturados
- **Para o domínio TCU**: cada resposta deve citar o KEY do acórdão usado

### 4. **Comparável**
- Mesmas queries em todas as versões
- Mesmas métricas avaliadas (qualitativas + Hit@K + MRR)
- Formato exportável (CSV, JSON) para análise cruzada

### Estrutura de um Benchmark Manual

```
Query → Embedding → Retrieval (k=5) → Ranking → LLM → Resposta
         ↓          ↓                           ↓
      Tempo      Chunks                     Tokens
      (ms)      Recuperados               Gerados
                Scores
                KEYs do TCU
```

Cada ponto medido permite diagnóstico de gargalos.


In [ ]:
# Célula 2: Inicializar pipeline RAG sobre Acórdãos do TCU
# - Embeddings:  bge-m3   via Ollama  (fallback: HuggingFace local)
# - LLM:         Groq API             (fallback 1: Ollama local | fallback 2: simulado)
# - Vector store: FAISS in-memory
# - Corpus: CSV de Acórdãos TCU 2026

import os, time, json, csv, re, requests, warnings
from typing import Any, Dict, List
from pathlib import Path
from datetime import datetime
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
warnings.filterwarnings('ignore')
load_dotenv()

OLLAMA_BASE_URL    = os.getenv('OLLAMA_BASE_URL',    'http://localhost:11434')
OLLAMA_LLM_MODEL   = os.getenv('OLLAMA_LLM_MODEL',   'llama3.1:8b')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL', 'bge-m3')
GROQ_API_KEY       = os.getenv('GROQ_API_KEY', '')
GROQ_LLM_MODEL     = os.getenv('GROQ_LLM_MODEL', 'llama-3.1-8b-instant')
# 👉 ALUNO: caminho do CSV de acórdãos TCU 2026
DATASET_DIR = Path("../datasets").resolve()
CSV_PATH = DATASET_DIR / "acordao-completo-2026.csv"

print('\n' + '='*60)
print('INICIALIZAÇÃO DO PIPELINE RAG — ACÓRDÃOS TCU 2026')
print('='*60 + '\n')

# ========== 1. EMBEDDINGS: BGE-M3 via Ollama (fallback HuggingFace) ==========
print(f'[1/4] Carregando embeddings BGE-M3...')
start_embed = time.time()
embedding_model = None

try:
    from langchain_ollama import OllamaEmbeddings
    requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=3).raise_for_status()
    embedding_model = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL, base_url=OLLAMA_BASE_URL)
    dim_real = len(embedding_model.embed_query('teste'))
    embed_backend = f'Ollama({OLLAMA_EMBED_MODEL})'
    print(f'   ✓ {embed_backend} | dim={dim_real}')
except Exception as e:
    print(f'   ⚠️  Ollama embeddings indisponível ({e}). Caindo para HuggingFace.')
    from langchain_community.embeddings import HuggingFaceEmbeddings
    embedding_model = HuggingFaceEmbeddings(
        model_name='BAAI/bge-m3',
        model_kwargs={'trust_remote_code': True, 'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True},
    )
    dim_real = len(embedding_model.embed_query('teste'))
    embed_backend = 'HuggingFace(BAAI/bge-m3)'
    print(f'   ✓ {embed_backend} | dim={dim_real}')

embed_time = time.time() - start_embed
print(f'   ⏱️  setup embeddings: {embed_time:.2f}s\n')

# ========== 2. CORPUS: ACÓRDÃOS TCU LIDOS DO CSV ==========
print('[2/4] Carregando acórdãos TCU do CSV...')

def _limpar_html(texto: str) -> str:
    """Remove tags HTML/XML do texto do campo ACORDAO."""
    return re.sub(r'<[^>]+>', ' ', texto).strip()

def carregar_acordaos(csv_path: str, keys_alvo: set = None) -> List[Document]:
    """
    Lê o CSV de acórdãos e retorna LangChain Documents.

    Args:
        csv_path: caminho do CSV (separador "|")
        keys_alvo: se fornecido, filtra apenas os KEYs solicitados.
                   Se None, carrega TODOS os acórdãos (modo produção).

    👉 ALUNO: o baseline inicial usa keys_alvo para indexar apenas os
    5 TOP-1 esperados (cenário ideal). Para o experimento de escala,
    troque keys_alvo=None e indexe o CSV completo.
    """
    csv.field_size_limit(10_000_000)
    docs = []
    with open(csv_path, encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='|')
        for row in reader:
            key = row.get('KEY', '').strip()
            if keys_alvo is not None and key not in keys_alvo:
                continue
            texto = _limpar_html(row.get('ACORDAO', ''))
            if not texto:
                continue
            docs.append(Document(
                page_content=texto,
                metadata={
                    'source':        key,   # campo padrão LangChain
                    'key':           key,
                    'titulo':        row.get('TITULO', '').strip(),
                    'tipo_processo': row.get('TIPOPROCESSO', '').strip(),
                    'colegiado':     row.get('COLEGIADO', '').strip(),
                    'relator':       row.get('RELATOR', '').strip(),
                    'data_sessao':   row.get('DATASESSAO', '').strip(),
                    'num_acordao':   row.get('NUMACORDAO', '').strip(),
                    'ano_acordao':   row.get('ANOACORDAO', '').strip(),
                    'assunto':       row.get('ASSUNTO', '').strip()[:300],
                },
            ))
    return docs

# 👉 ALUNO: KEYs dos 5 documentos TOP-1 esperados (gabarito).
#    Esse é o "modo baseline didático" — corpus pequeno e controlado.
#    Mude para `KEYS_TOP1 = None` para indexar o CSV completo (5.998 docs).
KEYS_TOP1 = {
    'ACORDAO-COMPLETO-2765836',  # Q1 — Auditoria Operacional Saúde 2025
    'ACORDAO-COMPLETO-2756361',  # Q2 — Representação CREA-MG / LMS
    'ACORDAO-COMPLETO-2752522',  # Q3 — Representação PMDF Engenharia
    'ACORDAO-COMPLETO-2758568',  # Q4 — TCE INSS Genésio Almeida Vinente
    'ACORDAO-COMPLETO-2763267',  # Q5 — Monitoramento INSS Acórdão 634/2025
}

corpus_docs = carregar_acordaos(CSV_PATH, keys_alvo=KEYS_TOP1)
print(f'   ✓ {len(corpus_docs)} acórdãos carregados do CSV')
for d in corpus_docs:
    print(f'      • {d.metadata["key"]} — {d.metadata["titulo"][:70]}')

print('\n   Criando índice FAISS...')
start_faiss = time.time()
vectorstore = FAISS.from_documents(corpus_docs, embedding_model)
faiss_time = time.time() - start_faiss
print(f'   ✓ FAISS criado em {faiss_time:.2f}s\n')

# ========== 3. LLM: Groq (com Fallback Dinâmico para Ollama e Simulado) ==========
print('[3/4] Configurando LLM (Estratégia: Groq com Fallback para Ollama)...')
llm = None
llm_mode = None

# TENTATIVA 1: Groq
if GROQ_API_KEY:
    try:
        from langchain_groq import ChatGroq
        llm = ChatGroq(
            model=GROQ_LLM_MODEL,
            groq_api_key=GROQ_API_KEY,
            temperature=0,
            max_tokens=512
        )
        _ = llm.invoke('OK em uma palavra:')
        llm_mode = f'Groq Cloud ({GROQ_LLM_MODEL})'
        print(f'   ✓ {llm_mode} carregado como LLM Principal!')
    except Exception as e:
        print(f'   ⚠️ Falha ao inicializar Groq ({e}). Tentando Ollama...')

# TENTATIVA 2: Ollama local
if llm is None:
    try:
        from langchain_ollama import ChatOllama
        requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=3).raise_for_status()
        llm = ChatOllama(
            model=OLLAMA_LLM_MODEL,
            base_url=OLLAMA_BASE_URL,
            temperature=0,
            num_predict=512,
        )
        _ = llm.invoke('OK em uma palavra:')
        llm_mode = f'Ollama Local ({OLLAMA_LLM_MODEL}) [Fallback]'
        print(f'   ✓ {llm_mode}')
    except Exception as e:
        print(f'   ⚠️ Ollama LLM indisponível ({e}). Ativando modo simulado.')

# TENTATIVA 3: Simulado
class SimulatedLLM:
    def __init__(self):
        self.model = 'simulated-deterministic'
        self.temperature = 0
    def invoke(self, prompt: str) -> Any:
        return type('obj', (object,), {'content': 'Resposta simulada (modo offline): ' + str(prompt)[:100] + '...'})()

if llm is None:
    llm = SimulatedLLM()
    llm_mode = 'Simulado (offline) — Ambos os provedores falharam'
    print(f'   ✓ {llm_mode}')

print(f'   ✓ LLM ativo: {llm_mode}\n')

# ========== 4. PROMPT TEMPLATE TCU ==========
print('[4/4] Configurando template de prompt para acórdãos TCU...\n')

# 👉 Prompt adaptado do domínio penal/segurança para CONTROLE EXTERNO/TCU.
#    A regra de citação agora exige o KEY do acórdão, permitindo verificar
#    se o LLM está realmente usando o documento recuperado.
template_tcu = """Você é um assistente especializado em controle externo e jurisprudência do Tribunal de Contas da União (TCU).

### INSTRUÇÃO CRÍTICA: SEMPRE CITE O ACÓRDÃO DE ORIGEM

Você receberá um contexto (acórdãos do TCU) e uma pergunta. Responda APENAS com informações presentes nos documentos do contexto.

**REGRA 1**: Se a informação não estiver no contexto, responda exatamente:
             "Não há informação suficiente no contexto para responder."
**REGRA 2**: Cada afirmação DEVE ter citação no formato:
             [Acórdão NUMERO/ANO – COLEGIADO | KEY: IDENTIFICADOR]
**REGRA 3**: Diferencie claramente determinações, recomendações e ciências.
**REGRA 4**: Cite valores monetários, números de subitens (ex.: 9.2.1) e              artigos de lei (ex.: art. 57 da Lei 8.443/1992) com precisão.
**REGRA 5**: Se a pergunta envolver múltiplos acórdãos, organize a resposta              por acórdão de origem.

### CONTEXTO (Acórdãos Recuperados)

{context}

### PERGUNTA

{question}

### RESPOSTA

Responda em português jurídico, estruturado, com todas as fontes citadas."""

prompt_template = PromptTemplate(
    input_variables=['context', 'question'],
    template=template_tcu,
)

print('='*60)
print('PIPELINE RAG INICIALIZADO COM SUCESSO')
print('='*60)
print(f'\nConfigurações do baseline:')
print(f'  • Embeddings: {embed_backend} (dim={dim_real})')
print(f'  • Índice: FAISS ({len(corpus_docs)} acórdãos)')
print(f'  • LLM: {llm_mode}')
print(f'  • K (chunks recuperados): 5')
print(f'  • Temperatura: 0 (determinístico)')
print(f'  • Tempo total de setup: {embed_time + faiss_time:.2f}s')
print('='*60)


## As 5 Queries de Baseline

As queries foram selecionadas para cobrir **5 tipos diferentes de necessidade informacional** sobre jurisprudência do TCU, cada uma desafiando aspectos diferentes do RAG:

| ID | Categoria | Tipo de Processo | Desafio | Esperado do Naive RAG |
|---|---|---|---|---|
| Q1 | **Recomendações de Auditoria** | Relatório de Auditoria (RA) | Extrair subitens normativos (9.1.x) de um relatório longo | Bom (texto literal no acórdão) |
| Q2 | **Causa de Decisão** | Representação (REPR) | Identificar motivo específico de desclassificação em licitação | Excelente (causa explícita) |
| Q3 | **Valor + Justificativa** | Representação (REPR) | Recuperar valor monetário preciso e fundamentação | Médio (texto verboso) |
| Q4 | **Condenação + Base Legal** | TCE | Vincular pena à base legal e diferenciar réu homônimo | Médio-Fraco (há duplicata no corpus) |
| Q5 | **Análise Discriminada** | Monitoramento (MON) | Distinguir categorias dentro do mesmo documento | Fraco (requer parsing fino) |

### Estratégia de Construção
- **Q1-Q2**: Queries que o Naive RAG deve responder bem (validação positiva)
- **Q3**: Query com valor numérico específico — testa precisão de extração
- **Q4**: Query com **armadilha de ambiguidade** — há um segundo acórdão (`ACORDAO-COMPLETO-2737698`) sobre o mesmo réu (Genésio Almeida Vinente) na 2ª Câmara. Quando o CSV completo for indexado, esse documento pode "confundir" o ranking e expor a fraqueza do Naive RAG.
- **Q5**: Query analítica que exige distinguir determinações "em cumprimento" vs "cumpridas" dentro do mesmo acórdão

### Gabarito (TOP-1 esperado por query)

| Query | TOP-1 esperado (KEY) | Acórdão |
|---|---|---|
| Q1 | `ACORDAO-COMPLETO-2765836` | Acórdão 1199/2026 – Plenário |
| Q2 | `ACORDAO-COMPLETO-2756361` | Acórdão de Relação 1393/2026 – 2ª Câmara |
| Q3 | `ACORDAO-COMPLETO-2752522` | Acórdão de Relação 587/2026 – Plenário |
| Q4 | `ACORDAO-COMPLETO-2758568` | Acórdão 1132/2026 – Plenário |
| Q5 | `ACORDAO-COMPLETO-2763267` | Acórdão de Relação 985/2026 – Plenário |


In [ ]:
# Célula 3: Definir as 5 Queries de Baseline (Acórdãos TCU)
# Estrutura: id, categoria, texto, resposta esperada, KEY do TOP-1, documentos relevantes

queries_baseline = [
    {
        "id": "Q1",
        "categoria": "Recomendações de Auditoria",
        "texto": (
            "Quais recomendações o TCU fez ao Ministério da Saúde em relação aos "
            "indicadores do Plano Plurianual (PPA) na auditoria operacional sobre "
            "os recursos federais aplicados na função Saúde em 2025?"
        ),
        "resposta_esperada": (
            "Explicitar premissas técnicas das metas do PPA (subitem 9.1.1); "
            "promover revisão progressiva dos indicadores para ampliar indicadores "
            "de resultado e impacto (9.1.2); adotar mecanismos de priorização e "
            "registro técnico em casos de restrição orçamentária (9.1.3)."
        ),
        "top1_esperado": "ACORDAO-COMPLETO-2765836",
        "documentos_relevantes": ["ACORDAO-COMPLETO-2765836"],
    },
    {
        "id": "Q2",
        "categoria": "Causa de Decisão",
        "texto": (
            "Por qual motivo a empresa representante foi desclassificada no Pregão "
            "Eletrônico PE-90032-A/2025 do CREA-MG para contratação de plataforma LMS?"
        ),
        "resposta_esperada": (
            "Descumprimento objetivo do item 9.3 do edital: a licitante recusou-se "
            "deliberadamente a enviar as credenciais de acesso à Comissão de Contratação "
            "dentro do prazo de 2 horas, condicionando a entrega a manifestação prévia "
            "do TCU. Como o ato é vinculado, a desclassificação tornou-se obrigatória."
        ),
        "top1_esperado": "ACORDAO-COMPLETO-2756361",
        "documentos_relevantes": ["ACORDAO-COMPLETO-2756361"],
    },
    {
        "id": "Q3",
        "categoria": "Valor + Justificativa",
        "texto": (
            "Qual foi o valor do suposto dano apontado na representação contra a "
            "Polícia Militar do Distrito Federal no Pregão Eletrônico 90042/2025, "
            "e qual a decisão do TCU quanto à necessidade de instaurar tomada de "
            "contas especial?"
        ),
        "resposta_esperada": (
            "Dano alegado: R$ 7.726,67 (diferença entre proposta vencedora e do "
            "representante), valor abaixo do limite mínimo para instauração de TCE. "
            "O TCU optou por não atuar corretivamente de forma imediata, pois a PMDF "
            "já tratava os fatos internamente; bastou encaminhar a decisão à unidade."
        ),
        "top1_esperado": "ACORDAO-COMPLETO-2752522",
        "documentos_relevantes": ["ACORDAO-COMPLETO-2752522"],
    },
    {
        "id": "Q4",
        "categoria": "Condenação + Base Legal",
        "texto": (
            "Genésio Almeida Vinente foi condenado em tomada de contas especial por "
            "concessão irregular de benefício assistencial. Qual a multa aplicada e "
            "com base em qual artigo da Lei 8.443/1992?"
        ),
        "resposta_esperada": (
            "Multa aplicada com fundamento no art. 57 da Lei 8.443/1992, c/c art. 267 "
            "do Regimento Interno do TCU. Réu declarado revel (art. 12, § 3º da Lei "
            "8.443/1992); contas julgadas irregulares por concessão de benefício "
            "assistencial sem atender aos critérios da LOAS."
        ),
        "top1_esperado": "ACORDAO-COMPLETO-2758568",
        "documentos_relevantes": ["ACORDAO-COMPLETO-2758568"],
    },
    {
        "id": "Q5",
        "categoria": "Análise Discriminada",
        "texto": (
            "No monitoramento do Acórdão 634/2025-Plenário referente ao INSS, quais "
            "determinações foram consideradas apenas 'em cumprimento' (e não totalmente "
            "cumpridas), e qual unidade técnica foi autorizada a continuar o monitoramento?"
        ),
        "resposta_esperada": (
            "Em cumprimento: 9.2.1, 9.2.3, 9.2.4, 9.2.5, 9.2.7, 9.2.8, 9.2.9 e 9.2.11. "
            "Cumpridas: 9.2.2, 9.2.6 e 9.2.10. Unidade técnica autorizada a continuar "
            "o monitoramento: AudBenefícios (Unidade de Auditoria Especializada em "
            "Previdência, Assistência e Trabalho), em até 90 dias."
        ),
        "top1_esperado": "ACORDAO-COMPLETO-2763267",
        "documentos_relevantes": ["ACORDAO-COMPLETO-2763267"],
    },
]

# ========== APRESENTAÇÃO DAS QUERIES ==========
print("\n" + "="*70)
print("QUERIES DE BASELINE — ACÓRDÃOS TCU 2026")
print("="*70 + "\n")

import pandas as pd

df_queries = pd.DataFrame([
    {
        "ID": q["id"],
        "Categoria": q["categoria"],
        "Query": q["texto"][:60] + "...",
        "TOP-1 Esperado": q["top1_esperado"],
    }
    for q in queries_baseline
])

print(df_queries.to_string(index=False))
print("\n" + "="*70 + "\n")

for q in queries_baseline:
    print(f"\n┌─ QUERY {q['id']}: {q['categoria'].upper()}")
    print(f"├─ Texto: {q['texto']}")
    print(f"├─ Resposta Esperada: {q['resposta_esperada']}")
    print(f"└─ Documento TOP-1: {q['top1_esperado']}")

print("\n" + "="*70)
print(f"TOTAL: {len(queries_baseline)} queries definidas para baseline")
print("="*70)


## Execução e Coleta de Respostas

Nesta seção, executamos cada query através do pipeline RAG completo:

```
Query → [EMBEDDING] → [RETRIEVAL] → [LLM] → Response
         ↓ tempo      ↓ chunks      ↓ tempo
         medido       medido        medido
                      ↓ KEYs
                      Hit@1 / Hit@K / RR
```

Coletamos:
- Chunks recuperados, KEYs e scores de similaridade
- Tempo de embedding, retrieval e LLM
- Resposta completa do LLM
- **Métricas de recuperação por query**: Hit@1, Hit@K e Reciprocal Rank (RR)


In [ ]:
# Célula 4: Executar as 5 queries e coletar respostas + métricas de recuperação
# Função: executar_query_completa() com medição de tempo, artefatos e Hit@K/MRR

from tqdm import tqdm

K_RETRIEVAL = 5  # número de chunks recuperados por consulta

def calcular_metricas_recuperacao(keys_recuperados: List[str], top1_esperado: str) -> Dict:
    """
    Calcula Hit@1, Hit@K e Reciprocal Rank.

    - hit@1  : KEY na posição 1 é o esperado?
    - hit@K  : KEY esperado aparece nos K resultados?
    - rank   : posição (1-indexed); 0 se ausente
    - rr     : 1/rank ou 0 — usado para compor o MRR
    """
    rank = 0
    for i, k in enumerate(keys_recuperados, 1):
        if k == top1_esperado:
            rank = i
            break
    return {
        "hit_at_1": rank == 1,
        "hit_at_k": rank > 0,
        "rank":     rank,
        "rr":       (1.0 / rank) if rank > 0 else 0.0,
    }


def executar_query_completa(query_info: Dict, vectorstore, llm, prompt_template, embedding_model) -> Dict[str, Any]:
    """
    Executa uma query completa através do pipeline RAG e coleta artefatos + métricas.

    Args:
        query_info: dict com id, texto, categoria, top1_esperado, etc.
        vectorstore: índice FAISS
        llm: modelo de linguagem
        prompt_template: template de prompt TCU
        embedding_model: modelo de embeddings

    Returns:
        Dicionário com respostas, tempos e métricas de recuperação.
    """
    result = {
        "id": query_info["id"],
        "categoria": query_info["categoria"],
        "query_texto": query_info["texto"],
        "resposta_esperada": query_info["resposta_esperada"],
        "top1_esperado": query_info["top1_esperado"],
        "timestamp": datetime.now().isoformat()
    }

    # ===== PASSO 1: EMBEDDING DA QUERY =====
    inicio_embedding = time.time()
    query_embedding = embedding_model.embed_query(query_info["texto"])
    tempo_embedding = time.time() - inicio_embedding
    result["tempo_embedding_ms"] = round(tempo_embedding * 1000, 2)
    result["tamanho_embedding"] = len(query_embedding)

    # ===== PASSO 2: RETRIEVAL (busca no FAISS) =====
    inicio_retrieval = time.time()
    # Compatibilidade entre versões de LangChain (similarity_search_with_score vs _scores)
    if hasattr(vectorstore, "similarity_search_with_score"):
        docs_recuperados = vectorstore.similarity_search_with_score(query_info["texto"], k=K_RETRIEVAL)
    else:
        docs_recuperados = vectorstore.similarity_search_with_scores(query_info["texto"], k=K_RETRIEVAL)
    tempo_retrieval = time.time() - inicio_retrieval
    result["tempo_retrieval_ms"] = round(tempo_retrieval * 1000, 2)
    result["num_chunks_recuperados"] = len(docs_recuperados)

    # Armazenar chunks com metadados, KEYs e scores
    chunks_info = []
    keys_recuperados = []
    contexto_para_llm = ""

    for i, (doc, score) in enumerate(docs_recuperados, 1):
        key = doc.metadata.get("key", doc.metadata.get("source", "N/A"))
        keys_recuperados.append(key)
        chunk_entry = {
            "rank": i,
            "score_similaridade": round(float(score), 4),
            "key": key,
            "titulo": doc.metadata.get("titulo", "N/A"),
            "tipo_processo": doc.metadata.get("tipo_processo", "N/A"),
            "colegiado": doc.metadata.get("colegiado", "N/A"),
            "conteudo_preview": doc.page_content[:200] + "..." if len(doc.page_content) > 200 else doc.page_content,
        }
        chunks_info.append(chunk_entry)

        # Contexto para o LLM — incluindo metadados estruturados
        contexto_para_llm += (
            f"--- DOCUMENTO {i} ---\n"
            f"KEY: {key}\n"
            f"Título: {doc.metadata.get('titulo', 'N/A')}\n"
            f"Tipo: {doc.metadata.get('tipo_processo', 'N/A')}\n"
            f"Colegiado: {doc.metadata.get('colegiado', 'N/A')}\n"
            f"Relator: {doc.metadata.get('relator', 'N/A')}\n"
            f"Data Sessão: {doc.metadata.get('data_sessao', 'N/A')}\n\n"
            f"TEXTO DO ACÓRDÃO:\n{doc.page_content[:3000]}\n\n"
        )

    result["chunks"] = chunks_info
    result["keys_recuperados"] = keys_recuperados
    result["contexto_recuperado"] = contexto_para_llm

    # ===== MÉTRICAS DE RECUPERAÇÃO (Hit@1, Hit@K, RR) =====
    metricas = calcular_metricas_recuperacao(keys_recuperados, query_info["top1_esperado"])
    result.update(metricas)

    # ===== PASSO 3: GERAÇÃO COM LLM =====
    inicio_llm = time.time()
    prompt_final = prompt_template.format(context=contexto_para_llm, question=query_info["texto"])

    if isinstance(llm, SimulatedLLM):
        resposta_llm = f"Resposta simulada para {query_info['id']}: {query_info['resposta_esperada']}"
        tokens_gerados = len(resposta_llm.split())
    else:
        try:
            msg = llm.invoke(prompt_final)
            resposta_llm = msg.content if hasattr(msg, "content") else str(msg)
            tokens_gerados = len(resposta_llm.split())
        except Exception as e:
            resposta_llm = f"[Erro ao chamar LLM: {e}]"
            tokens_gerados = 0

    tempo_llm = time.time() - inicio_llm
    result["tempo_llm_ms"] = round(tempo_llm * 1000, 2)
    result["resposta"] = resposta_llm
    result["tokens_estimados"] = tokens_gerados
    result["tempo_total_ms"] = round((tempo_embedding + tempo_retrieval + tempo_llm) * 1000, 2)

    return result


print("\n" + "="*70)
print("EXECUTANDO QUERIES DE BASELINE — ACÓRDÃOS TCU")
print("="*70 + "\n")

resultados_queries = []
for query_info in tqdm(queries_baseline, desc="Processando queries", unit="query"):
    resultado = executar_query_completa(
        query_info, vectorstore, llm, prompt_template, embedding_model
    )
    resultados_queries.append(resultado)

    print(f"\n{'='*70}")
    print(f"[{resultado['id']}] {resultado['categoria'].upper()}")
    print(f"{'='*70}")
    print(f"\nQuery: {resultado['query_texto']}")
    print(f"\nResposta Esperada: {resultado['resposta_esperada']}")
    print(f"\n📊 RECUPERAÇÃO (k={K_RETRIEVAL}):")
    for chunk in resultado["chunks"]:
        marker = "✅" if chunk["key"] == resultado["top1_esperado"] else "  "
        print(f"   {marker} #{chunk['rank']} {chunk['key']} (score={chunk['score_similaridade']})")
    print(f"\n📈 hit@1={resultado['hit_at_1']} | hit@{K_RETRIEVAL}={resultado['hit_at_k']} | "
          f"rank={resultado['rank']} | RR={resultado['rr']:.3f}")
    print(f"\n🤖 Resposta Gerada:\n{resultado['resposta'][:400]}...\n")
    print(f"⏱️  Métricas de tempo:")
    print(f"   • Embedding: {resultado['tempo_embedding_ms']:.2f}ms")
    print(f"   • Retrieval: {resultado['tempo_retrieval_ms']:.2f}ms")
    print(f"   • LLM:       {resultado['tempo_llm_ms']:.2f}ms")
    print(f"   • TOTAL:     {resultado['tempo_total_ms']:.2f}ms")
    print(f"   • Tokens estimados: {resultado['tokens_estimados']}")

print(f"\n{'='*70}")
print(f"EXECUÇÃO CONCLUÍDA: {len(resultados_queries)} queries processadas")

# Resumo das métricas de recuperação
hits1 = sum(r["hit_at_1"] for r in resultados_queries)
hitsk = sum(r["hit_at_k"] for r in resultados_queries)
mrr = sum(r["rr"] for r in resultados_queries) / len(resultados_queries)
print(f"\n📊 MÉTRICAS DE RECUPERAÇÃO AGREGADAS:")
print(f"   • Hit@1:  {hits1}/{len(resultados_queries)} ({hits1/len(resultados_queries)*100:.1f}%)")
print(f"   • Hit@{K_RETRIEVAL}: {hitsk}/{len(resultados_queries)} ({hitsk/len(resultados_queries)*100:.1f}%)")
print(f"   • MRR:    {mrr:.3f}")
print(f"{'='*70}")


## Avaliação Qualitativa Manual

Agora avaliamos cada resposta nas **5 dimensões críticas para RAG jurídico aplicado a acórdãos do TCU**:

### Dimensão 1: Relevância do Contexto Recuperado (retrieval_relevance)
- **5**: TOP-1 recuperado é o acórdão esperado E os demais chunks são acórdãos correlatos
- **3**: Acórdão esperado aparece, mas não no TOP-1 (apareceu nas demais posições)
- **1**: Acórdão esperado não está entre os K recuperados

### Dimensão 2: Completude da Resposta (answer_completeness)
- **5**: Contém TODAS as informações esperadas (subitens, valores, datas, base legal)
- **3**: Parcial — informação principal presente, faltam detalhes (ex.: cita art. 57 mas omite Lei 8.443)
- **1**: Incompleta ou superficial

### Dimensão 3: Fidelidade ao Contexto (faithfulness)
- **5**: Toda informação é rastreável aos acórdãos recuperados; sem alucinação
- **3**: Maioria rastreável, 1-2 pontos vagos ou inferências não-explícitas
- **1**: Contém valores, números de acórdão ou base legal inventados (alucinação grave)

### Dimensão 4: Qualidade das Citações (citation_quality)
- **5**: Cada afirmação cita `[Acórdão N/ANO – COLEGIADO | KEY: ...]` no formato correto
- **3**: Cita o número do acórdão mas omite o KEY ou o colegiado
- **1**: Sem citações ou citações genéricas ("o acórdão diz...")

### Dimensão 5: Utilidade Jurídica (legal_utility)
- **5**: Auditor/advogado pode usar diretamente em parecer ou nota técnica
- **3**: Útil como ponto de partida, requer verificação no acórdão original
- **1**: Não seria utilizado sem revisão extensiva

> **Importante:** os scores abaixo estão **pré-preenchidos com valores típicos** de um Naive RAG sobre corpus reduzido (5 docs). Após executar suas próprias queries, **revise cada score com base nas respostas reais geradas** pelo seu pipeline.


In [ ]:
# Célula 5: Framework de avaliação qualitativa com scoring pré-preenchido
# Demonstração com scores realistas e espaços para o aluno completar

# Definição das dimensões de avaliação
dimensoes_avaliacao = [
    {"id": "retrieval_relevance", "nome": "Relevância do Contexto Recuperado", "peso": 1.0},
    {"id": "answer_completeness", "nome": "Completude da Resposta", "peso": 1.0},
    {"id": "faithfulness", "nome": "Fidelidade ao Contexto / Anti-Alucinação", "peso": 1.2},
    {"id": "citation_quality", "nome": "Qualidade das Citações", "peso": 1.1},
    {"id": "legal_utility", "nome": "Utilidade Jurídica", "peso": 1.0}
]

# ===== SCORES PRÉ-PREENCHIDOS PARA DEMONSTRAÇÃO =====
# Perfil típico do Naive RAG sobre acórdãos TCU (corpus reduzido):
# - Recuperação: tende a 5 (corpus pequeno, FAISS acerta TOP-1)
# - Completude: varia conforme a complexidade da query
# - Fidelidade: tende a 4-5 (BGE-M3 + temperatura 0 reduz alucinação)
# - Citações: tende a 3-4 (LLM cita Acórdão mas frequentemente omite KEY)
# - Utilidade: cai nas queries analíticas (Q4 com homônimo, Q5 com categorias)
scores_demonstracao = {
    "Q1": {  # Recomendações de auditoria — texto literal e estruturado
        "retrieval_relevance": 5,
        "answer_completeness": 5,
        "faithfulness": 5,
        "citation_quality": 4,
        "legal_utility": 5
    },
    "Q2": {  # Causa de desclassificação — narrativa direta
        "retrieval_relevance": 5,
        "answer_completeness": 5,
        "faithfulness": 5,
        "citation_quality": 4,
        "legal_utility": 5
    },
    "Q3": {  # Valor monetário específico — testa precisão
        "retrieval_relevance": 5,
        "answer_completeness": 4,
        "faithfulness": 4,
        "citation_quality": 3,
        "legal_utility": 4
    },
    "Q4": {  # Condenação + base legal — armadilha com homônimo
        "retrieval_relevance": 4,
        "answer_completeness": 3,
        "faithfulness": 4,
        "citation_quality": 3,
        "legal_utility": 3
    },
    "Q5": {  # Análise discriminada — categorias dentro do mesmo doc
        "retrieval_relevance": 3,
        "answer_completeness": 2,
        "faithfulness": 3,
        "citation_quality": 2,
        "legal_utility": 2
    }
}

# Construir dataframe de avaliação
df_avaliacao = pd.DataFrame()

for query_id, scores_query in scores_demonstracao.items():
    linha = {"Query": query_id}
    for dim in dimensoes_avaliacao:
        linha[dim["nome"]] = scores_query[dim["id"]]
    scores_list = list(scores_query.values())
    linha["Score Médio"] = round(sum(scores_list) / len(scores_list), 2)
    df_avaliacao = pd.concat([df_avaliacao, pd.DataFrame([linha])], ignore_index=True)

print("\n" + "="*120)
print("MATRIZ DE AVALIAÇÃO QUALITATIVA — NAIVE RAG BASELINE (ACÓRDÃOS TCU)")
print("Escala: 1 (fraco) a 5 (excelente)")
print("="*120 + "\n")

print(df_avaliacao.to_string(index=False))

print("\n" + "="*120)
print("LEGENDA E INSTRUÇÕES")
print("="*120)
print("""
Cada dimensão deve ser preenchida manualmente após revisar a resposta:

1. RELEVÂNCIA DO CONTEXTO RECUPERADO (Escala 1-5)
   5 = TOP-1 é o acórdão esperado e os demais são correlatos
   3 = Acórdão esperado aparece mas fora do TOP-1
   1 = Acórdão esperado não está nos K resultados
   Critério: O FAISS trouxe o acórdão certo na posição certa?

2. COMPLETUDE DA RESPOSTA (Escala 1-5)
   5 = Resposta cobre TODOS os pontos esperados (subitens, valores, datas)
   3 = Parcial — falta detalhe (ex.: cita art. 57 mas omite Lei 8.443)
   1 = Incompleta ou superficial
   Critério: A resposta responde a TODOS os aspectos da pergunta?

3. FIDELIDADE AO CONTEXTO (Escala 1-5)
   5 = Toda informação é rastreável aos acórdãos recuperados
   3 = Maioria rastreável, 1-2 pontos vagos
   1 = Contém valores ou números de acórdão inventados (alucinação)
   Critério: É possível verificar cada afirmação no texto do acórdão?

4. QUALIDADE DAS CITAÇÕES (Escala 1-5)
   5 = Cada afirmação cita [Acórdão N/ANO – COLEGIADO | KEY: ...]
   3 = Cita número do acórdão mas omite KEY ou colegiado
   1 = Sem citações ou citações genéricas
   Critério: Posso rastrear a origem de cada informação até o KEY?

5. UTILIDADE JURÍDICA (Escala 1-5)
   5 = Auditor pode usar a resposta diretamente em parecer
   3 = Útil como ponto de partida, requer verificação
   1 = Não seria utilizado sem revisão extensiva
   Critério: A resposta serve para fundamentar uma manifestação técnica?

PREENCHIDO COM VALORES DE DEMONSTRAÇÃO: 'Score Médio' mostra a média das 5 dimensões.
ALUNO: Revise cada score após ler as respostas geradas e ajuste conforme necessário.
""")
print("="*120)


## Calculando as Métricas Agregadas do Baseline

Combinamos duas famílias de métricas:

1. **Qualitativas (manuais)**: scores 1-5 nas 5 dimensões → score médio por dimensão e score geral ponderado
2. **Recuperação (automáticas)**: Hit@1, Hit@K e MRR a partir dos KEYs comparados ao gabarito

A combinação permite diagnosticar onde está o problema: se Hit@1 é alto mas o score geral é baixo, o gargalo é o **LLM/geração**; se Hit@1 é baixo, o gargalo é o **retrieval**.


In [ ]:
# Célula 6: Calcular métricas agregadas do baseline (scores_dimensao, score_geral, métricas de recuperação)

print("\n" + "="*70)
print("MÉTRICAS AGREGADAS DO BASELINE")
print("="*70 + "\n")

# ===== 1. Score médio por DIMENSÃO (média entre as 5 queries) =====
dimensoes_nomes = [d["nome"] for d in dimensoes_avaliacao]
scores_dimensao = {}
for dim_nome in dimensoes_nomes:
    scores_dimensao[dim_nome] = round(df_avaliacao[dim_nome].mean(), 2)

print("📊 SCORE MÉDIO POR DIMENSÃO:")
for nome, score in scores_dimensao.items():
    barras = "█" * int(score) + "░" * (5 - int(score))
    print(f"   {barras}  {score:.2f}/5.0  {nome}")

# ===== 2. Score GERAL ponderado (usando o peso de cada dimensão) =====
soma_pesos = sum(d["peso"] for d in dimensoes_avaliacao)
score_geral = sum(scores_dimensao[d["nome"]] * d["peso"] for d in dimensoes_avaliacao) / soma_pesos
print(f"\n🎯 SCORE GERAL PONDERADO: {score_geral:.2f}/5.0")

# ===== 3. Métricas de RECUPERAÇÃO agregadas =====
hits1 = sum(r["hit_at_1"] for r in resultados_queries)
hitsk = sum(r["hit_at_k"] for r in resultados_queries)
mrr = sum(r["rr"] for r in resultados_queries) / len(resultados_queries)
hit1_pct = hits1 / len(resultados_queries) * 100
hitk_pct = hitsk / len(resultados_queries) * 100

print(f"\n📈 MÉTRICAS DE RECUPERAÇÃO:")
print(f"   • Hit@1:  {hits1}/{len(resultados_queries)} ({hit1_pct:.1f}%)")
print(f"   • Hit@{K_RETRIEVAL}: {hitsk}/{len(resultados_queries)} ({hitk_pct:.1f}%)")
print(f"   • MRR:    {mrr:.3f}")

# ===== 4. Tempos médios =====
tempo_emb_medio = sum(r["tempo_embedding_ms"] for r in resultados_queries) / len(resultados_queries)
tempo_ret_medio = sum(r["tempo_retrieval_ms"] for r in resultados_queries) / len(resultados_queries)
tempo_llm_medio = sum(r["tempo_llm_ms"] for r in resultados_queries) / len(resultados_queries)
tempo_total_medio = sum(r["tempo_total_ms"] for r in resultados_queries) / len(resultados_queries)

print(f"\n⏱️  TEMPOS MÉDIOS (ms):")
print(f"   • Embedding: {tempo_emb_medio:.2f}ms")
print(f"   • Retrieval: {tempo_ret_medio:.2f}ms")
print(f"   • LLM:       {tempo_llm_medio:.2f}ms")
print(f"   • TOTAL:     {tempo_total_medio:.2f}ms")

# ===== 5. Diagnóstico automático =====
print(f"\n🔍 DIAGNÓSTICO AUTOMÁTICO:")
if hit1_pct >= 80 and score_geral < 4.0:
    print("   ⚠️  Hit@1 alto MAS score geral baixo → gargalo está no LLM/geração.")
    print("       → Ações: melhorar prompt, usar modelo maior, adicionar reranking semântico.")
elif hit1_pct < 60:
    print("   ⚠️  Hit@1 baixo → gargalo está no RETRIEVAL.")
    print("       → Ações: testar outro embedding, reranking BM25 + denso, query expansion.")
else:
    print("   ✓ Pipeline balanceado. Próximo passo: indexar o CSV completo e re-medir.")

print("\n" + "="*70)


## Visualizações do Baseline

Quatro gráficos compõem o painel diagnóstico:

1. **Radar Chart** — perfil de desempenho do Naive RAG nas 5 dimensões
2. **Heatmap** — matriz Query × Dimensão (identifica células fracas)
3. **Decomposição de tempos** — embedding vs retrieval vs LLM por query
4. **Hit@1 vs Score Médio** — combina recuperação e qualidade por query


In [ ]:
# Célula 7: Visualizações do baseline com 4 gráficos
# Radar chart, heatmap, decomposição de tempos e Hit@1 vs Score

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from math import pi

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 12)
plt.rcParams['font.size'] = 10

fig = plt.figure(figsize=(16, 12))

# ===== GRÁFICO 1: RADAR CHART (Spider) =====
ax1 = plt.subplot(2, 2, 1, projection='polar')

categories = [dim[:22] + '...' if len(dim) > 22 else dim for dim in dimensoes_nomes]
values = [scores_dimensao[dim] for dim in dimensoes_nomes]
values += values[:1]

angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
angles += angles[:1]

ax1.plot(angles, values, 'o-', linewidth=2, color='#2E86AB', label='Naive RAG (TCU)')
ax1.fill(angles, values, alpha=0.25, color='#2E86AB')
ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(categories, size=8)
ax1.set_ylim(0, 5)
ax1.set_yticks([1, 2, 3, 4, 5])
ax1.set_title('Perfil de Desempenho: Naive RAG sobre Acórdãos TCU', size=12, weight='bold', pad=20)
ax1.grid(True)
ax1.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

# ===== GRÁFICO 2: HEATMAP Query × Dimensão =====
ax2 = plt.subplot(2, 2, 2)
heatmap_data = df_avaliacao[['Query'] + dimensoes_nomes].set_index('Query')
sns.heatmap(heatmap_data, annot=True, fmt='g', cmap='RdYlGn', vmin=1, vmax=5,
            ax=ax2, cbar_kws={'label': 'Score'}, linewidths=0.5)
ax2.set_title('Matriz: Query × Dimensão de Avaliação', size=12, weight='bold')
ax2.set_xlabel('Dimensão de Qualidade', fontweight='bold')
ax2.set_ylabel('Query de Teste', fontweight='bold')

# ===== GRÁFICO 3: DECOMPOSIÇÃO DE TEMPOS =====
ax3 = plt.subplot(2, 2, 3)

queries_ids = [r['id'] for r in resultados_queries]
tempos_emb = [r['tempo_embedding_ms'] for r in resultados_queries]
tempos_ret = [r['tempo_retrieval_ms'] for r in resultados_queries]
tempos_llm_list = [r['tempo_llm_ms'] for r in resultados_queries]

x = np.arange(len(queries_ids))
width = 0.25

ax3.bar(x - width, tempos_emb, width, label='Embedding', color='#A23B72')
ax3.bar(x, tempos_ret, width, label='Retrieval', color='#F18F01')
ax3.bar(x + width, tempos_llm_list, width, label='LLM', color='#2E86AB')

ax3.set_ylabel('Tempo (ms)', fontweight='bold')
ax3.set_xlabel('Query', fontweight='bold')
ax3.set_title('Decomposição de Tempo por Etapa do Pipeline', size=12, weight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(queries_ids)
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# ===== GRÁFICO 4: HIT@1 vs SCORE MÉDIO POR QUERY =====
ax4 = plt.subplot(2, 2, 4)

scores_por_query = df_avaliacao['Score Médio'].tolist()
hit1_por_query = [1.0 if r['hit_at_1'] else 0.0 for r in resultados_queries]
rr_por_query = [r['rr'] for r in resultados_queries]

x = np.arange(len(queries_ids))
width = 0.35

bars1 = ax4.bar(x - width/2, scores_por_query, width, label='Score Médio (1-5)',
                color='#2E86AB', edgecolor='black')
bars2 = ax4.bar(x + width/2, [s * 5 for s in rr_por_query], width,
                label='RR × 5 (escala visual)', color='#F18F01', edgecolor='black')

ax4.set_ylabel('Pontuação', fontweight='bold')
ax4.set_xlabel('Query', fontweight='bold')
ax4.set_title('Qualidade vs Recuperação por Query', size=12, weight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(queries_ids)
ax4.set_ylim(0, 5.5)
ax4.legend()
ax4.grid(axis='y', alpha=0.3)

for bar, val in zip(bars1, scores_por_query):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val:.2f}', ha='center', fontsize=8, fontweight='bold')
for bar, val in zip(bars2, rr_por_query):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val:.2f}', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/baseline_tcu_visualizacoes.png', dpi=150, bbox_inches='tight')
print("\n✓ Gráficos salvos em /tmp/baseline_tcu_visualizacoes.png")
plt.show()

print("\n" + "="*70)
print("VISUALIZAÇÕES DO BASELINE GERADAS")
print("="*70)


## Exportando o Baseline em 3 Formatos

Geramos três artefatos para versionar o baseline:

- **CSV**: scores por query + métricas de recuperação (importação fácil em planilha)
- **Excel** (multi-aba): respostas geradas + scores + KEYs recuperados
- **JSON**: snapshot completo do baseline (metadados, config, métricas) — fonte canônica para comparação futura


In [ ]:
# Célula 8: Exportar baseline em CSV, Excel e JSON
import json
from datetime import datetime
import pandas as pd
import numpy as np

print("\n" + "="*70)
print("EXPORTANDO BASELINE EM 3 FORMATOS")
print("="*70 + "\n")

# ===== FORMATO 1: CSV =====
df_export_csv = pd.DataFrame([
    {
        'query_id': row['Query'],
        'categoria': next(r['categoria'] for r in resultados_queries if r['id'] == row['Query']),
        'top1_esperado': next(r['top1_esperado'] for r in resultados_queries if r['id'] == row['Query']),
        'top1_recuperado': next(r['keys_recuperados'][0] if r['keys_recuperados'] else 'N/A'
                                for r in resultados_queries if r['id'] == row['Query']),
        'hit_at_1': next(r['hit_at_1'] for r in resultados_queries if r['id'] == row['Query']),
        'hit_at_k': next(r['hit_at_k'] for r in resultados_queries if r['id'] == row['Query']),
        'rank': next(r['rank'] for r in resultados_queries if r['id'] == row['Query']),
        'rr': next(r['rr'] for r in resultados_queries if r['id'] == row['Query']),
        'score_medio': row['Score Médio'],
        'retrieval_relevance': row['Relevância do Contexto Recuperado'] if 'Relevância do Contexto Recuperado' in row else row.get('retrieval_relevance', 3),
        'answer_completeness': row['Completude da Resposta'] if 'Completude da Resposta' in row else row.get('answer_completeness', 3),
        'faithfulness': row['Fidelidade ao Contexto / Anti-Alucinação'] if 'Fidelidade ao Contexto / Anti-Alucinação' in row else row.get('faithfulness', 3),
        'citation_quality': row['Qualidade das Citações'] if 'Qualidade das Citações' in row else row.get('citation_quality', 3),
        'legal_utility': row['Utilidade Jurídica'] if 'Utilidade Jurídica' in row else row.get('legal_utility', 3),
        'tempo_total_ms': next(r['tempo_total_ms'] for r in resultados_queries if r['id'] == row['Query']),
    }
    for _, row in df_avaliacao.iterrows()
])

csv_path = 'baseline_naive_rag_tcu.csv'
df_export_csv.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"✓ CSV exportado: {csv_path}")
print(f"  Linhas: {len(df_export_csv)}\n")

# ===== FORMATO 2: EXCEL (multi-aba) =====
xlsx_path = 'baseline_tcu.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    # Aba 1: Respostas + tempos + métricas de recuperação
    df_respostas = pd.DataFrame([
        {
            'Query ID':         r['id'],
            'Categoria':        r['categoria'],
            'Query':            r['query_texto'],
            'TOP-1 Esperado':   r.get('top1_esperado', 'N/A'),
            'TOP-1 Recuperado': r['keys_recuperados'][0] if r.get('keys_recuperados') else 'N/A',
            'Hit@1':            r.get('hit_at_1', False),
            'Hit@K':            r.get('hit_at_k', False),
            'Rank':             r.get('rank', 0),
            'RR':               round(r.get('rr', 0), 3),
            'Resposta LLM':     r['resposta'][:1000],
            'Resposta Esperada': r['resposta_esperada'],
            'Tempo Total (ms)': r['tempo_total_ms'],
            'Tokens Estimados': r['tokens_estimados'],
        }
        for r in resultados_queries
    ])
    df_respostas.to_excel(writer, sheet_name='Respostas', index=False)
    df_avaliacao.to_excel(writer, sheet_name='Scores Qualitativos', index=False)
    df_export_csv.to_excel(writer, sheet_name='Métricas Consolidadas', index=False)

print(f"✓ Excel exportado: {xlsx_path}")
print(f"  Abas: Respostas, Scores Qualitativos, Métricas Consolidadas\n")

# ===== FORMATO 3: JSON (snapshot canônico) =====

# Recuperando valores globais com fallbacks seguros caso não definidos
csv_corpus_str = str(globals().get('CSV_PATH', 'baseline_tcu.csv'))
docs_indexados = int(globals().get('corpus_docs', []).__len__() or len(df_export_csv))
modo_corpus = 'reduzido (TOP-1 esperados)' if globals().get('KEYS_TOP1') else 'completo'

baseline_json = {
    'metadata': {
        'aula':           'Aula 2 - Lab 5',
        'dominio':        'Acórdãos do TCU 2026',
        'data_criacao':   datetime.now().isoformat(),
        'versao':         '0.0',
        'csv_corpus':     csv_corpus_str, # 🟢 CORREÇÃO: Forçando conversão de Path para String pura
        'docs_indexados': docs_indexados,
        'modo_corpus':    modo_corpus,
    },
    'configuracao': {
        'embeddings':   str(globals().get('embed_backend', 'bge-m3')),
        'embed_dim':    int(globals().get('dim_real', 1024)),
        'llm':          str(globals().get('llm_mode', 'Groq/Ollama')),
        'k_retrieval':  int(globals().get('K_RETRIEVAL', 5)),
        'temperatura':  0,
        'num_queries':  len(resultados_queries),
    },
    'metricas_recuperacao': {
        'hit_at_1':     float(globals().get('hits1', 0) / len(resultados_queries)) if globals().get('hits1') else 0.0,
        'hit_at_k':     float(globals().get('hitsk', 0) / len(resultados_queries)) if globals().get('hitsk') else 0.0,
        'mrr':          round(float(globals().get('mrr', 0)), 3),
    },
    'metricas_qualitativas': {
        'score_geral_ponderado': round(float(globals().get('score_geral', 0)), 2),
        'scores_dimensao':       {str(k): round(float(v), 2) for k, v in globals().get('scores_dimensao', {}).items()},
    },
    'tempos_medios_ms': {
        'embedding': round(float(globals().get('tempo_emb_medio', 0)), 2),
        'retrieval': round(float(globals().get('tempo_ret_medio', 0)), 2),
        'llm':       round(float(globals().get('tempo_llm_medio', 0)), 2),
        'total':     round(float(globals().get('tempo_total_medio', 0)), 2),
    },
    'resultados_por_query': [
        {
            'id':                str(r['id']),
            'categoria':         str(r['categoria']),
            'query':             str(r['query_texto']),
            'top1_esperado':     str(r.get('top1_esperado', 'N/A')),
            'keys_recuperados':  [str(k) for k in r.get('keys_recuperados', [])],
            'hit_at_1':          bool(r.get('hit_at_1', False)),
            'hit_at_k':          bool(r.get('hit_at_k', False)),
            'rank':              int(r.get('rank', 0)),
            'rr':                round(float(r.get('rr', 0)), 3),
            'tempo_total_ms':    float(r['tempo_total_ms']),
            'resposta_llm':      str(r['resposta']),
        }
        for r in resultados_queries
    ],
}

json_path = 'baseline_tcu.json'

# Usamos um tratamento padrão personalizado (default=str) no dump como barreira de segurança definitiva
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(baseline_json, f, indent=2, ensure_ascii=False, default=str)

print(f"✓ JSON exportado: {json_path}")
print("\n" + "="*70)
print("EXPORTAÇÃO CONCLUÍDA")
print("="*70)
print(f"\nArtefatos do baseline v0.0 disponíveis na raiz do projeto:")
print(f"  • baseline_naive_rag_tcu.csv")
print(f"  • baseline_tcu.xlsx")
print(f"  • baseline_tcu.json")
print(f"  • baseline_tcu_visualizacoes.png")

## Análise de Fraquezas e Roadmap

A célula final identifica as **3 dimensões mais fracas** do baseline e mapeia cada uma a uma aula futura do curso. O roadmap deve ser revisitado ao final da aula 12 para validar se as melhorias foram alcançadas.


In [ ]:
# Célula 9: Análise de fraquezas e mapeamento para aulas futuras

print("\n" + "="*80)
print("ANÁLISE DE FRAQUEZAS: ROADMAP DE MELHORIAS (AULA 3-12)")
print("="*80 + "\n")

dimensoes_ranking = [
    ("Relevância do Contexto Recuperado",        scores_dimensao.get("Relevância do Contexto Recuperado", 3)),
    ("Completude da Resposta",                   scores_dimensao.get("Completude da Resposta", 3)),
    ("Fidelidade ao Contexto / Anti-Alucinação", scores_dimensao.get("Fidelidade ao Contexto / Anti-Alucinação", 3)),
    ("Qualidade das Citações",                   scores_dimensao.get("Qualidade das Citações", 3)),
    ("Utilidade Jurídica",                       scores_dimensao.get("Utilidade Jurídica", 3))
]

# Mapa de melhorias por dimensão (aulas futuras)
mapa_melhorias = {
    "Relevância do Contexto Recuperado": "Aula 3 (Reranking) + Aula 7 (Query Expansion) + Aula 10 (HyDE)",
    "Completude da Resposta": "Aula 5 (Compressão de Contexto) + Aula 8 (Self-RAG)",
    "Fidelidade ao Contexto / Anti-Alucinação": "Aula 8 (Self-RAG) + Aula 11 (Validação com Crítica)",
    "Qualidade das Citações": "Aula 4 (Prompt Engineering Avançado) + Aula 9 (Structured Output)",
    "Utilidade Jurídica": "Aula 6 (Chunking Semântico) + Aula 12 (Avaliação RAGAS)"
}

sorted_dims_weak = sorted(dimensoes_ranking, key=lambda x: x[1])

print("📉 TOP 3 DIMENSÕES MAIS FRACAS DO BASELINE:\n")
for rank, (nome, score) in enumerate(sorted_dims_weak[:3], 1):
    print(f"   #{rank}. {nome}: {score:.2f}/5.0")
    print(f"        → Plano: {mapa_melhorias.get(nome, 'N/A')}\n")

print(f"🎯 SCORE GERAL DO BASELINE: {score_geral:.2f}/5.0")
print(f"🎯 META (após todas as aulas): 4.0+/5.0")
print(f"📈 MELHORIA ESPERADA: +{max(0, 4.0 - score_geral):.2f} pontos\n")

# ===== Análise específica de RECUPERAÇÃO =====
print("📊 ANÁLISE DE RECUPERAÇÃO:\n")
print(f"   • Hit@1 atual:  {hit1_pct:.1f}%  (meta produção: ≥ 70% com corpus completo)")
print(f"   • Hit@{K_RETRIEVAL} atual: {hitk_pct:.1f}%  (meta produção: ≥ 90% com corpus completo)")
print(f"   • MRR atual:    {mrr:.3f}  (meta produção: ≥ 0.80)")

if KEYS_TOP1 is not None:
    print(f"\n   ⚠️  ATENÇÃO: o baseline atual usa corpus REDUZIDO (apenas {len(corpus_docs)} acórdãos).")
    print(f"       Para validar produção, reexecute com KEYS_TOP1 = None (indexar 5.998 docs).")
    print(f"       Espera-se queda em Hit@1 — essa é a métrica que as aulas 3-12 vão atacar.")

print("\n" + "="*80)
print("📋 PRÓXIMOS EXPERIMENTOS SUGERIDOS PARA O ALUNO")
print("="*80)
print("""
EXPERIMENTO 1 — Escala do corpus
   Altere KEYS_TOP1 = None na Célula 2 e re-execute todo o notebook.
   → O CSV completo (5.998 acórdãos) será indexado.
   → Compare Hit@1 antes/depois. Q4 deve cair (homônimo aparece).

EXPERIMENTO 2 — Variação de K
   Mude K_RETRIEVAL para 1, 3, 10 e re-execute.
   → Plote Hit@K em função de K. Onde a curva satura?

EXPERIMENTO 3 — Chunking
   Acórdãos do TCU são longos. Atualmente cada acórdão é 1 documento.
   → Aplique RecursiveCharacterTextSplitter(chunk_size=1000, overlap=200).
   → Reindexe e compare MRR. Atenção: o KEY do acórdão deve permanecer
     no metadado de cada chunk para Hit@1 ainda funcionar.

EXPERIMENTO 4 — Trocar embedding
   Substitua BGE-M3 por all-MiniLM-L6-v2 (modelo menor/mais rápido).
   → Compare Hit@1 e latência. O ganho de velocidade compensa a queda?

EXPERIMENTO 5 — Análise da Q4 (homônimo)
   Indexe o CSV completo. Procure manualmente:
     - ACORDAO-COMPLETO-2758568 (Plenário, gabarito)
     - ACORDAO-COMPLETO-2737698 (2ª Câmara, mesmo réu)
   → Qual o TOP-1 retornado para a Q4? Como reformular a pergunta
     para forçar o ranking correto? Isso prepara a aula 7 (Query Expansion).
""")
print("="*80)
print("FIM DO NOTEBOOK — BASELINE TCU PRONTO PARA AULA 3")
print("="*80)
